## Name: Mustafa Erdem
## Date : 04/20/2026
## Exercise : Final Project  
###  Newark Liberty International Airport (EWR) – United Airlines Flight Analysis (2025)

This project analyzes United Airlines flight data specifically for departures from Newark Liberty International Airport (EWR) in 2025. The focus is on flight delays, scheduling accuracy, and operational performance.

In [1]:
import pandas as pd

In [2]:
print("Pandas version:", pd.__version__)

Pandas version: 2.3.3


## Data Cleaning: Removing Corrupted Rows

Before loading the dataset into pandas, the file was cleaned by removing the last two rows, which were causing parsing errors.

This step is necessary because the dataset contains corrupted or incomplete lines at the end of the file, which can break CSV parsing.

After removing these rows, the cleaned data is saved into a new file (`clean_file.csv`) and then used for further analysis.

In [3]:
with open("Detailed_Statistics_Departures.csv", "r") as f:
    lines = f.readlines()

with open("clean_file.csv", "w") as f:
    f.writelines(lines[:-2])

##  Removed Rows

The following rows were removed from the dataset because they were causing parsing errors. 

In [4]:
for line in lines[-2:]:
    print(line)



 SOURCE: Bureau of Transportation Statistics


In [5]:
df = pd.read_csv("clean_file.csv", skiprows=7)  ## The first 7 rows are skipped because they contain metadata,
                                                ## and the actual data table begins afterward.

# Removed Rows
The following rows were removed from the dataset.

In [6]:
count = 0;
for line in lines:
    if count < 7:
        count += 1
        print(line)

Detailed Statistics Departures  

Origin Airport: Newark, NJ: Newark Liberty International (EWR)

Airline: United Airlines Inc. (UA)

Month(s): January, February, March, April, May, June, July, August, September, October, November, and December

Day(s): 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30 and 31

Year(s): 2025





##  Dataset Size

The cleaned dataset contains **66,747 flight records**, confirming that the data was successfully loaded and is ready for analysis.

In [7]:
df.shape 

(66747, 17)

In [8]:
df.columns  ## The column names were reviewed for clarity and usability.
            ## Long or complex column names may be shortened or standardized to improve readability and make analysis easier.

Index(['Carrier Code', 'Date (MM/DD/YYYY)', 'Flight Number', 'Tail Number',
       'Destination Airport', 'Scheduled departure time',
       'Actual departure time', 'Scheduled elapsed time (Minutes)',
       'Actual elapsed time (Minutes)', 'Departure delay (Minutes)',
       'Wheels-off time', 'Taxi-Out time (Minutes)', 'Delay Carrier (Minutes)',
       'Delay Weather (Minutes)', 'Delay National Aviation System (Minutes)',
       'Delay Security (Minutes)', 'Delay Late Aircraft Arrival (Minutes)'],
      dtype='object')

In [9]:
df.info() ## Only the “Tail Number” column contains missing values, which I try to handled in later preprocessing steps.

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 66747 entries, 0 to 66746
Data columns (total 17 columns):
 #   Column                                    Non-Null Count  Dtype 
---  ------                                    --------------  ----- 
 0   Carrier Code                              66747 non-null  object
 1   Date (MM/DD/YYYY)                         66747 non-null  object
 2   Flight Number                             66747 non-null  int64 
 3   Tail Number                               65850 non-null  object
 4   Destination Airport                       66747 non-null  object
 5   Scheduled departure time                  66747 non-null  object
 6   Actual departure time                     66747 non-null  object
 7   Scheduled elapsed time (Minutes)          66747 non-null  int64 
 8   Actual elapsed time (Minutes)             66747 non-null  int64 
 9   Departure delay (Minutes)                 66747 non-null  int64 
 10  Wheels-off time                           6674

##  Descriptive Statistics Summary

On average, flights have a departure delay of about 15 minutes, while taxi-out time averages around 26 minutes. Most delay categories such as weather, security, and national aviation system delays have very low median values (often 0), indicating that these delays do not occur frequently.

However, the maximum values show significant outliers in several columns (for example, very large departure delays and carrier delays), suggesting that while most flights experience minor disruptions, a small number of flights suffer extreme delays. This difference between average and maximum values highlights variability in airline performance and the impact of exceptional cases.

In [10]:
display(df.describe().round(3))

,Flight Number,Scheduled elapsed time (Minutes),Actual elapsed time (Minutes),Departure delay (Minutes),Taxi-Out time (Minutes),Delay Carrier (Minutes),Delay Weather (Minutes),Delay National Aviation System (Minutes),Delay Security (Minutes),Delay Late Aircraft Arrival (Minutes)
count,66747.000,66747.000,66747.000,66747.000,66747.000,66747.000,66747.000,66747.000,66747.000,66747.000
mean,1582.867,223.546,214.620,15.345,26.279,4.918,0.926,5.103,0.002,8.314
std,697.150,93.832,97.658,54.233,16.384,24.433,14.673,18.952,0.526,34.999
min,42.000,65.000,0.000,-32.000,0.000,0.000,0.000,0.000,0.000,0.000
25%,1184.000,160.000,152.000,-6.000,17.000,0.000,0.000,0.000,0.000,0.000
50%,1625.000,193.000,192.000,-3.000,22.000,0.000,0.000,0.000,0.000,0.000
75%,2151.000,288.000,288.000,11.000,29.000,0.000,0.000,0.000,0.000,0.000
max,3012.000,687.000,721.000,1240.000,379.000,899.000,871.000,654.000,136.000,1112.000


In [11]:
display(df.head())

,Carrier Code,Date (MM/DD/YYYY),Flight Number,Tail Number,Destination Airport,Scheduled departure time,Actual departure time,Scheduled elapsed time (Minutes),Actual elapsed time (Minutes),Departure delay (Minutes),Wheels-off time,Taxi-Out time (Minutes),Delay Carrier (Minutes),Delay Weather (Minutes),Delay National Aviation System (Minutes),Delay Security (Minutes),Delay Late Aircraft Arrival (Minutes)
0,UA,01/01/2025,211,N828UA,BNA,17:29,17:21,157,156,-8,17:46,25,0,0,0,0,0
1,UA,01/01/2025,212,N69819,TPA,20:25,20:34,185,213,9,21:34,60,9,0,28,0,0
2,UA,01/01/2025,293,N38446,RSW,06:45,06:38,195,179,-7,06:52,14,0,0,0,0,0
3,UA,01/01/2025,363,N76054,HNL,09:05,09:09,684,664,4,09:33,24,0,0,0,0,0
4,UA,01/01/2025,403,N27252,FLL,10:00,10:11,197,186,11,10:31,20,0,0,0,0,0


In [12]:
display(df.tail())

,Carrier Code,Date (MM/DD/YYYY),Flight Number,Tail Number,Destination Airport,Scheduled departure time,Actual departure time,Scheduled elapsed time (Minutes),Actual elapsed time (Minutes),Departure delay (Minutes),Wheels-off time,Taxi-Out time (Minutes),Delay Carrier (Minutes),Delay Weather (Minutes),Delay National Aviation System (Minutes),Delay Security (Minutes),Delay Late Aircraft Arrival (Minutes)
66742,UA,12/31/2025,2678,N894UA,JAX,13:00,12:55,162,136,-5,13:12,17,0,0,0,0,0
66743,UA,12/31/2025,2679,N41140,LAX,10:15,10:11,380,353,-4,10:27,16,0,0,0,0,0
66744,UA,12/31/2025,2691,N17306,MIA,20:15,20:12,202,172,-3,20:31,19,0,0,0,0,0
66745,UA,12/31/2025,2692,N12021,LAX,08:40,08:36,365,353,-4,08:56,20,0,0,0,0,0
66746,UA,12/31/2025,2856,N37504,DFW,20:29,20:23,249,219,-6,20:46,23,0,0,0,0,0


In [13]:
display(df.sample())

,Carrier Code,Date (MM/DD/YYYY),Flight Number,Tail Number,Destination Airport,Scheduled departure time,Actual departure time,Scheduled elapsed time (Minutes),Actual elapsed time (Minutes),Departure delay (Minutes),Wheels-off time,Taxi-Out time (Minutes),Delay Carrier (Minutes),Delay Weather (Minutes),Delay National Aviation System (Minutes),Delay Security (Minutes),Delay Late Aircraft Arrival (Minutes)
33666,UA,07/07/2025,1342,N37465,IAH,11:55,12:23,225,194,28,12:40,17,0,0,0,0,0
